In [3]:
import pandas as pd
import scanpy as sc
from utils.io import read_anndata
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from dask import array as da
from tqdm.dask import TqdmCallback

from utils.io import read_anndata

zebrafish_adata = read_anndata(
    '/Users/seohyon/Downloads/Wagner2018.h5ad',
    # X='X',
    obs='obs',
    var='var',
    obsm='obsm',
)

cengen_adata = read_anndata(
    '/Users/seohyon/Downloads/Hammarlund2018.h5ad',
    # X='X',
    obs='obs',
    var='var',
    obsm='obsm',
)

dask: False, backed: False
Read slot "obs", store as "obs"...
Read slot "var", store as "var"...
Read slot "obsm", store as "obsm"...
dask: False, backed: False
Read slot "obs", store as "obs"...
Read slot "var", store as "var"...
Read slot "obsm", store as "obsm"...


In [6]:
print(zebrafish_adata)
print(cengen_adata)

AnnData object with n_obs × n_vars = 26022 × 25258
    obs: 'lab', 'sample_label', 'hpf', 'cell_type', 'labels', 'is_train', 'n_counts', 'batch', 'size_factors'
    var: 'n_cells', 'feature_name', 'hvg', 'hvg_score'
    obsm: 'X_pca'
AnnData object with n_obs × n_vars = 100955 × 22469
    obs: 'dropbox_id', 'counts', 'experiment_code', 'cell_type', 'tissue', 'n_counts', 'batch', 'size_factors'
    var: 'n_cells', 'feature_name', 'hvg', 'hvg_score'
    obsm: 'X_pca'


In [13]:
# zebrafish_adata.obs["tissue"].nunique() - no tissue obs
cengen_adata.obs["tissue"].nunique()

11

In [9]:
print(zebrafish_adata.obs["cell_type"].nunique())
print(cengen_adata.obs["cell_type"].nunique())

24
169


In [14]:
print(zebrafish_adata.obs["batch"].nunique())
print(cengen_adata.obs["batch"].nunique())

2
17


In [15]:
print(zebrafish_adata.obs["sample_label"].nunique())
print(cengen_adata.obs["experiment_code"].nunique())

60
17


In [16]:
zebrafish_adata.obs[["batch", "sample_label"]].drop_duplicates().value_counts("batch")

batch
Klein     32
Schier    28
Name: count, dtype: int64

In [17]:
zebrafish_adata.obs[["batch", "sample_label"]].drop_duplicates().value_counts("sample_label")

sample_label
SRR6890845_04hpf    1
SRR6890846_04hpf    1
SRR6890847_06hpf    1
SRR6890848_06hpf    1
SRR6890849_06hpf    1
SRR6890850_06hpf    1
SRR6890851_08hpf    1
SRR6890852_08hpf    1
SRR6890853_08hpf    1
SRR6890854_08hpf    1
SRR6890855_10hpf    1
SRR6890856_10hpf    1
SRR6890857_10hpf    1
SRR6890858_10hpf    1
SRR6890859_14hpf    1
SRR6890860_14hpf    1
SRR6890861_14hpf    1
SRR6890862_14hpf    1
SRR6890863_18hpf    1
SRR6890864_18hpf    1
SRR6890865_18hpf    1
SRR6890866_18hpf    1
SRR6890867_18hpf    1
SRR6890868_18hpf    1
SRR6890869_18hpf    1
SRR6890870_24hpf    1
SRR6890871_24hpf    1
SRR6890872_24hpf    1
SRR6890873_24hpf    1
SRR6890874_24hpf    1
SRR6890875_24hpf    1
SRR6890876_24hpf    1
ZF3S-DS5            1
ZF6S-DS5            1
ZF6S-DS5b           1
ZF30-DS5            1
ZF30-DS5b           1
ZF50-DS2            1
ZF50-DS3            1
ZF50-DS4            1
ZF50-DS4b           1
ZF60-DS2            1
ZF60-DS3            1
ZF60-DS4            1
ZF75-DS2           

In [18]:
cengen_adata.obs[["batch", "experiment_code"]].drop_duplicates().value_counts("batch")

batch
Pan-1           1
Pan-2           1
acr-2           1
ceh-28_dat-1    1
ceh-34          1
cho-1_1         1
cho-1_2         1
eat-4           1
ift-20          1
nlp-13_ceh-2    1
nmr-1           1
tph-1_ceh-10    1
unc-3           1
unc-47_1        1
unc-47_2        1
unc-53          1
unc-86          1
Name: count, dtype: int64

In [19]:
cengen_adata.obs[["batch", "experiment_code"]].drop_duplicates().value_counts("experiment_code")

experiment_code
Pan-1           1
Pan-2           1
acr-2           1
ceh-28_dat-1    1
ceh-34          1
cho-1_1         1
cho-1_2         1
eat-4           1
ift-20          1
nlp-13_ceh-2    1
nmr-1           1
tph-1_ceh-10    1
unc-3           1
unc-47_1        1
unc-47_2        1
unc-53          1
unc-86          1
Name: count, dtype: int64

In [20]:
batch_size = zebrafish_adata.obs["batch"].value_counts().sort_values(ascending=False)
mean = np.mean(batch_size)
std = np.std(batch_size)
min_val = batch_size.min()
max_val = batch_size.max()
print(f"Mean batch size: {mean:.2f}")
print(f"Standard deviation: {std:.2f}")
print(f"Minimum batch size: {min_val}")
print(f"Maximum batch size: {max_val}")
print(f"Coefficient of variance: {std/mean:.2f}")

Mean batch size: 13011.00
Standard deviation: 2403.00
Minimum batch size: 10608
Maximum batch size: 15414
Coefficient of variance: 0.18


In [21]:
batch_size = cengen_adata.obs["batch"].value_counts().sort_values(ascending=False)
mean = np.mean(batch_size)
std = np.std(batch_size)
min_val = batch_size.min()
max_val = batch_size.max()
print(f"Mean batch size: {mean:.2f}")
print(f"Standard deviation: {std:.2f}")
print(f"Minimum batch size: {min_val}")
print(f"Maximum batch size: {max_val}")
print(f"Coefficient of variance: {std/mean:.2f}")

Mean batch size: 5938.53
Standard deviation: 3067.44
Minimum batch size: 1669
Maximum batch size: 13139
Coefficient of variance: 0.52
